In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import gc
import os
import sys

In [3]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("env ready GPU-Cache is cleared")

env ready GPU-Cache is cleared


In [ ]:
proj_path = "/content/drive/MyDrive/llm_from_scratch/src"
output_path = "/content/drive/MyDrive/llm_from_scratch/text_generation_model"

sys.path.append(proj_path)
os.chdir(proj_path)
print(f"cwd switched to: {os.getcwd()}")


cwd switched to: /content/drive/MyDrive/llm_from_scratch/src


In [7]:
import torch
import tiktoken
from torch.utils.data import Dataset
from datasets import load_dataset

class DatasetProcessing(Dataset):
    def __init__(self, raw_dataset, max_length=512):
        # using our GPT-2 tokenizer matched to our custom architecture.
        self.encoding = tiktoken.get_encoding('gpt2')
        self.max_length = max_length
        self.raw_dataset = raw_dataset

    def __len__(self):
        return len(self.raw_dataset)

    def __getitem__(self, idx):
        item = self.raw_dataset[idx]
        conversations_list = item['conversations']

        #constructing the user and chatbot multi-turn conversation
        chat_history = ""
        user_demands = []

        for turn in conversations_list:
            speaker_role = turn['from']
            text_content = turn['value'].strip()

            if speaker_role == "system":
                continue # we keep input clean here if it is from system
            elif speaker_role == "human":
                chat_history += f"user: {text_content}\n"
                user_demands.append(text_content)
            elif speaker_role == "gpt":
                chat_history += f"assistant: {text_content}\n"

        raw_target_text = conversations_list[-1]['value'].strip()
        words = [w for w in raw_target_text.replace("\n", " ").split(" ") if w]

        # a meaningful slice of the output for the target title
        if len(words) > 7:
            title_target = " ".join(words[:5]).strip(".,!? ")
        else:
            title_target = " ".join(words).strip(".,!? ")

        user_demands_str = " ".join(user_demands).lower()
        if any(keyword in user_demands_str for keyword in ["how", "write", "code", "delete", "run", "fix"]):
            instruction = "Read the following chat log and provide a short title describing what the user wants or what is being discussed."
        else:
            instruction = "Read the following conversation and generate a short declarative title summarizing its core conceptual theme."

        # template structure split into prompt and target
        prompt_text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Conversation:\n{chat_history.strip()}\n\n"
            f"### Title:\n"
        )

        # encoding them separately to find the exact boundary
        prompt_tokens = self.encoding.encode(prompt_text)
        target_tokens = self.encoding.encode(title_target)

        if len(prompt_tokens) + len(target_tokens) > self.max_length:
            # calculating how many tokens we can allow the prompt to have
            available_prompt_slots = self.max_length - len(target_tokens)

            # truncating the last part of the conversation, keeping the vital target title intact
            prompt_tokens = prompt_tokens[:available_prompt_slots]

        #combine for input
        input_ids = prompt_tokens + target_tokens

        #masking the lables with -100 to remove them from being part in loss calcualtion
        labels = [-100] * len(prompt_tokens) + target_tokens


        return {
            'input_ids': input_ids,
            'labels': labels
        }

print("loading dataset from Hugging Face...")
shared_download = load_dataset("Open-Orca/SlimOrca", split="train")

train_slice = shared_download.select(range(0, 12000))
val_slice = shared_download.select(range(12000, 14000))

# creating raw dataset into preprocessed ready for training dataset:
train_dataset = DatasetProcessing(train_slice, max_length=512)
val_dataset = DatasetProcessing(val_slice, max_length=512)

print("data has been loaded successfully.")

In [8]:
import torch
from transformers.generation.utils import GenerationMixin
from gpt_model import GPTModel
from config import GPTConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#dynamically patching GPTModel to inherit GenerationMixin
if GenerationMixin not in GPTModel.__bases__:
    print("injecting GenerationMixin into GPTModel base structural layers...")
    GPTModel.__bases__ = (GPTModel.__bases__[0], GenerationMixin) + GPTModel.__bases__[1:]

CONFIG = GPTConfig()

# initialising model
model = GPTModel(CONFIG)

# loading weights from pretrained weights form .pth file
CORRECT_PRETRAINED_PATH = "/content/drive/MyDrive/llm_from_scratch/pretrained_weights.pth"
model.load_state_dict(torch.load(CORRECT_PRETRAINED_PATH, map_location=device))

print("model is initialized and weights are successfully loaded.")

injecting GenerationMixin into GPTModel base structural layers...
model is initialized and weights are successfully loaded.


In [9]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [10]:
from peft import LoraConfig, get_peft_model

def setup_lora_model(model):
    target_modules = ["W_query", "W_key", "W_value", "out_proj", "out_head"]

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        modules_to_save=None
    )
    return get_peft_model(model, lora_config)

model = setup_lora_model(model)
model.to(device)
print("LoRA configurations attached to weight channels.")


LoRA configurations attached to weight channels.


In [16]:
class DynamicDataCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        #to find maximum sequence length present in each specific batch
        batch_lens = [len(f['input_ids']) for f in features]
        max_batch_len = max(batch_lens)

        batch_input_ids = []
        batch_labels = []

        for f in features:
            inputs = f['input_ids']
            labels = f['labels']
            remainder = max_batch_len - len(inputs)

            # pading inputs with eot token and labels with -100
            padded_inputs = inputs + [self.pad_token_id] * remainder
            padded_labels = labels + [-100] * remainder

            batch_input_ids.append(padded_inputs)
            batch_labels.append(padded_labels)

        return {
            'input_ids': torch.tensor(batch_input_ids, dtype=torch.long),
            'labels': torch.tensor(batch_labels, dtype=torch.long)
        }

# initializing the collator using tiktoken eot value
data_collator = DynamicDataCollator(pad_token_id=train_dataset.encoding.eot_token)


In [ ]:
from transformers import TrainingArguments, Trainer

os.environ["WANDB_DISABLED"] = "true"

training_args = TrainingArguments(
    output_dir=output_path,
    num_train_epochs=2,
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    fp16=True,
    logging_steps=10,
    remove_unused_columns=False,
    save_strategy="no",
    save_total_limit=1,
    report_to="none",

    # eval settings:
    per_device_eval_batch_size=2,
    eval_steps=100
)

training statring ...


Step,Training Loss
10,70.628662
20,66.915088
30,66.629645
40,71.081213
50,63.520544
60,65.787518
70,64.564008
80,65.248645
90,59.034467
100,58.466608


In [ ]:
# tranining (and parameters adjustments)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator # tells trainer to use dynamic padding per batch
)

print("training statring ...")
trainer.train()
print("training completed.")

In [ ]:
# importing finetuned model


# merging LoRA weights with existing ones
merged_model = model.merge_and_unload()

#saving treined weights
final_save_path = "/content/drive/MyDrive/llm_from_scratch/text_generation_model/chat_title_generator_model.pth"
torch.save(merged_model.state_dict(), final_save_path)

print(f"successfully weights file generated at: {final_save_path}")


successfully weights file generated at: /content/drive/MyDrive/llm_from_scratch/text_generation_model/title_model/chat_title_generator_model_final.pth
